# Ecommerce data quality with PySpark and Great Expectations

An ecommerce platform receives customer and order extracts. A successful file delivery does not guarantee usable data: an order can refer to an unknown customer, contain an invalid amount, or repeat an earlier order ID.

We will stage **two tables**, `gx.stg_customers` and `gx.stg_orders`, validate them, quarantine rejected rows with reasons, and publish accepted Parquet data to HDFS only after quality gates pass.

**Flow:** synthetic source → HDFS staging → safe parsing → GX checks → row quarantine → batch gates → HDFS publication marker.

Run cells from top to bottom in a **WSL Python kernel**. All records are fictional. The deliberately faulty batch should fail initial validation; those expected failures are part of the walkthrough.


## 1. Prepare the WSL environment

Use Python 3.10 or 3.11, Java 17, PySpark 3.5.6, and GX Core 1.19.1. These are pinned lesson dependencies, not a request to upgrade your existing Spark installation. In a WSL terminal, create a dedicated environment if needed:

```bash
python -m pip install  great_expectations==1.19.1
```

Select that kernel from your WSL notebook editor. `JAVA_HOME` must point to Java 17. If `SPARK_HOME` is set, it must match the installed PySpark version; an unrelated system Spark installation can cause class-loading errors.

HDFS must already be running with its NameNode at `hdfs://localhost:9000`. The DataNode must also be reachable and your WSL user must have write access to `/user/<your-user>`. Port 9000 is the filesystem RPC endpoint, not a browser URL. Hive on port 9083 is not used. The SQL catalog is in memory: table registrations disappear when Spark stops, while HDFS files remain.


In [ ]:
import getpass
import json
import platform
import sys
from datetime import datetime, timezone
from uuid import uuid4

import great_expectations as gx
from pyspark.sql import SparkSession, Window, functions as F, types as T

assert platform.system() == "Linux", "Select a WSL Linux Python kernel."
assert (3, 10) <= sys.version_info[:2] <= (3, 11), "Use the documented Python 3.10/3.11 environment."
assert gx.__version__ == "1.19.1", "Use GX Core 1.19.1 for this walkthrough."

spark = (SparkSession.builder.master("local[2]")
    .appName("gx_ecommerce_quality")
    .config("spark.sql.catalogImplementation", "in-memory")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "true")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
assert spark.version == "3.5.6", f"Expected Spark 3.5.6, found {spark.version}"
assert spark.conf.get("spark.sql.catalogImplementation") == "in-memory", "Restart the kernel without Hive support."

HDFS = "hdfs://localhost:9000"
ROOT = f"{HDFS}/user/{getpass.getuser()}/gx"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid4().hex[:8]
RUN_ROOT = f"{ROOT}/runs/{RUN_ID}"
# Fixed business time makes freshness tests repeatable on any execution date.
AS_OF = "2026-09-14 12:00:00"
print({"python": sys.version.split()[0], "spark": spark.version, "gx": gx.__version__, "run": RUN_ROOT})


## 2. Check HDFS before creating data

**Problem:** a reachable NameNode alone does not prove that writes work. A missing DataNode or incorrect permissions can still break the pipeline.

**Solution:** write and read one small probe at this run's unique location. Every execution gets a new run directory; `errorifexists` prevents accidental overwrites. If this cell fails, check HDFS services and permissions before continuing.


In [ ]:
probe_path = f"{RUN_ROOT}/preflight"
spark.range(1).write.mode("errorifexists").parquet(probe_path)
assert spark.read.parquet(probe_path).count() == 1
spark.sql(f"CREATE DATABASE IF NOT EXISTS gx LOCATION '{ROOT}/warehouse'")
spark.sql("USE gx")
print("HDFS read/write and gx database are ready.")


## 3. Define two source contracts

Each customer row represents one customer. Each order row represents one order; `order_id` is globally unique. `source_row_id` identifies the physical source record, so repeated business keys still have distinct quarantine identities.

| Dataset | Main fields | Business contract |
|---|---|---|
| Customers | customer_id, email, country, signup_at, phone | Unique ID and email; allowed country; valid signup date; phone optional |
| Orders | order_id, customer_id, ordered_at, shipped_at, quantity, unit_price, discount, total, status, currency, ingested_at | Valid customer; sensible dates and amounts; total = quantity × unit_price − discount |

Raw values stay as strings, just like a text extract. We preserve them for diagnosis. Explicit schemas prevent one batch's values from silently changing inferred types. The separate source manifest says **15 customers and 24 orders** were delivered; these counts are independent constants, not computed from the received data.


In [ ]:
customer_cols = ["source_row_id", "customer_id", "email", "country", "signup_at", "phone"]
order_cols = ["source_row_id", "order_id", "customer_id", "ordered_at", "shipped_at",
              "quantity", "unit_price", "discount", "total", "status", "currency", "ingested_at"]
def string_schema(columns):
    return T.StructType([T.StructField(c, T.StringType(), True) for c in columns])

customers = [
    ("c01", "C001", "alice@example.com", "IN", "2026-01-01 09:00:00", "+919876543210"),
    ("c02", "C002", " BOB@EXAMPLE.COM ", " in ", "2026-01-02 09:00:00", None),
    ("c03", "C003", "cara@example.com", "US", "2026-01-03 09:00:00", None),
    ("c04", None, "noid@example.com", "IN", "2026-01-01 09:00:00", None),
    ("c05", "   ", "blank@example.com", "IN", "2026-01-01 09:00:00", None),
    ("c06", "C006", "missing-at.example.com", "IN", "2026-01-01 09:00:00", None),
    ("c07", "C007", "country@example.com", "ZZ", "2026-01-01 09:00:00", None),
    ("c08", "C008", "bad-date@example.com", "IN", "2026-02-30 09:00:00", None),
    ("c09", "C009", "future@example.com", "IN", "2027-01-01 09:00:00", None),
    ("c10", "C010", "first@example.com", "IN", "2026-01-01 09:00:00", None),
    ("c11", "C010", "second@example.com", "IN", "2026-01-01 09:00:00", None),
    ("c12", "C012", "shared@example.com", "IN", "2026-01-01 09:00:00", None),
    ("c13", "C013", " SHARED@example.com ", "IN", "2026-01-01 09:00:00", None),
    ("c14", "customer14", "format@example.com", "IN", "2026-01-01 09:00:00", None),
    ("c15", "C015", None, "IN", "2026-01-01 09:00:00", None),
]
base_order = dict(zip(order_cols, ["o01", "O001", "C001", "2026-09-13 10:00:00", None,
                                  "2", "100.00", "10.00", "190.00", "PAID", "INR", "2026-09-14 11:00:00"]))
def order(n, **changes):
    row = dict(base_order, source_row_id=f"o{n:02d}", order_id=f"O{n:03d}")
    row.update(changes)
    return tuple(row[c] for c in order_cols)

orders = [
    order(1),
    order(2, customer_id="C002", status=" shipped ", shipped_at="2026-09-14 09:00:00"),
    order(3, customer_id="C003", currency="USD", status="PENDING"),
    order(4, customer_id=None),
    order(5, customer_id="C999"),
    order(6, customer_id="C006"),  # Exists in staging, but customer itself is rejected.
    order(7, quantity="two"),
    order(8, unit_price="NaN"),
    order(9, unit_price="-5.00", total="-20.00"),
    order(10, quantity="0", discount="0.00", total="0.00"),
    order(11, total="999.00"),
    order(12, status="LOST"),
    order(13, currency="EUR"),
    order(14, ordered_at="2026-02-30 10:00:00"),
    order(15, ordered_at="2027-01-01 10:00:00"),
    order(16, status="SHIPPED", shipped_at=None),
    order(17, status="SHIPPED", shipped_at="2026-09-12 10:00:00"),
    order(18, ordered_at="2025-12-31 10:00:00"),
    order(19, ingested_at="2026-09-10 11:00:00"),
    order(20, order_id="O020"),
    order(21, order_id="O020"),
    order(22, discount="250.00", total="-50.00"),
    order(23, unit_price="100.001"),
    order(24, ingested_at="2026-09-13 09:00:00"),
]
SOURCE_MANIFEST = {"customers": 15, "orders": 24}
customer_raw = spark.createDataFrame(customers, string_schema(customer_cols))
order_raw = spark.createDataFrame(orders, string_schema(order_cols))

for name, df in [("customers", customer_raw), ("orders", order_raw)]:
    path = f"{RUN_ROOT}/staging/{name}"
    df.write.mode("errorifexists").parquet(path)
    # Only session-local metadata is repointed on a rerun; previous run files remain.
    spark.sql(f"CREATE TABLE IF NOT EXISTS gx.stg_{name} USING PARQUET LOCATION '{path}'")
    spark.sql(f"ALTER TABLE gx.stg_{name} SET LOCATION '{path}'")

customer_raw = spark.table("gx.stg_customers")
order_raw = spark.table("gx.stg_orders")
spark.sql("SHOW TABLES IN gx").show()
customer_raw.show(15, truncate=False)
order_raw.show(24, truncate=False)


## 4. Meet the GX objects and connect both datasets

A **Context** manages GX configuration. A Spark **Data Source** owns named **Assets**. A whole-DataFrame **Batch Definition** describes the batch supplied at runtime. An **Expectation** is one assertion; a **Suite** groups assertions; a **Validation Definition** pairs a suite with a batch. A **Checkpoint** runs a reusable validation workflow.

Our context is ephemeral for a self-contained run; later we explicitly save suite definitions and results in HDFS. GX evaluates data quality; it does not automatically repair or split the DataFrame.


In [ ]:
context = gx.get_context(mode="ephemeral")
source = context.data_sources.add_spark(name="local_spark")
customer_batch = source.add_dataframe_asset(name="customers").add_batch_definition_whole_dataframe("whole")
order_batch = source.add_dataframe_asset(name="orders").add_batch_definition_whole_dataframe("whole")


validation_results = {}
def validate(label, definition, df):
    result = definition.run(batch_parameters={"dataframe": df}, result_format="BASIC")
    validation_results[label] = result
    print(label, "passed:", result.success, result.statistics)
    for item in result.results:
        if not item.success:
            print("  FAIL", item.expectation_config.type,
                  item.expectation_config.kwargs.get("column", "<table>"),
                  "unexpected:", item.result.get("unexpected_count"))
    return result



## 5. Schema drift and missing deliveries are batch failures

**Problem:** row checks cannot detect a missing column before expressions reference it, an extra unapproved column, a changed Spark type, an empty file, or an incomplete delivery.

**Solution:** validate structure, source record identity, and manifest volume before business transformations. Dropping `currency` simulates schema drift; `limit(0)` simulates an empty delivery. The expected failures are asserted so the walkthrough continues.

Column order is strict in this example. If your source contract permits reordering, use a set-based schema policy instead. The Python type check supplements the GX name/order check; both must pass.


In [ ]:
def schema_validation(name, batch_definition, columns, expected_count):
    suite = context.suites.add(gx.ExpectationSuite(name=name))
    suite.add_expectation(gx.expectations.ExpectTableColumnsToMatchOrderedList(column_list=columns))
    suite.add_expectation(gx.expectations.ExpectTableRowCountToEqual(value=expected_count))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="source_row_id"))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="source_row_id"))
    return context.validation_definitions.add(gx.ValidationDefinition(name="validate_" + name, data=batch_definition, suite=suite))

customer_structure = schema_validation("customer_delivery", customer_batch, customer_cols, SOURCE_MANIFEST["customers"])
order_structure = schema_validation("order_delivery", order_batch, order_cols, SOURCE_MANIFEST["orders"])
def types_match(df, columns):
    return df.columns == columns and all(isinstance(field.dataType, T.StringType) for field in df.schema.fields)

customer_delivery_result = validate("customer_delivery", customer_structure, customer_raw)
order_delivery_result = validate("order_delivery", order_structure, order_raw)
assert customer_delivery_result.success and order_delivery_result.success
assert types_match(customer_raw, customer_cols) and types_match(order_raw, order_cols)
assert not validate("schema_drift_demo", order_structure, order_raw.drop("currency")).success
assert not validate("empty_delivery_demo", order_structure, order_raw.limit(0)).success
assert not types_match(order_raw.withColumn("quantity", F.lit(2)), order_cols)


## 6. Normalize safely and expose parsing failures

**Problem:** `" in "` and `"IN"` mean the same thing, but `"two"` is not a quantity. A plain cast can throw under ANSI mode. Casting money directly to two decimal places could also hide a source precision error by rounding.

**Solution:** preserve every source column as `raw_*`, trim and standardize approved fields, and use `try_cast` / `try_to_timestamp` so invalid values become null. Check the original money text against a two-decimal contract before trusting the decimal result. `NaN`, infinity, excess precision, and overflow cannot become accepted money.

We deliberately do not invent missing IDs or choose a duplicate winner. Null checks are separate from range checks because many GX value expectations exclude nulls from their denominator.


In [ ]:
def raw_copy(df, columns):
    return df.select("source_row_id", *[F.col(c).alias("raw_" + c) for c in columns if c != "source_row_id"])
def clean_text(raw_name):
    value = F.trim(F.col(raw_name))
    return F.when(F.length(value) > 0, value)
def timestamp(raw_name):
    return F.try_to_timestamp(F.col(raw_name), F.lit("yyyy-MM-dd HH:mm:ss"))

c = raw_copy(customer_raw, customer_cols)
for field in ["customer_id", "email", "country", "phone"]:
    c = c.withColumn(field, clean_text("raw_" + field))
c = (c.withColumn("email", F.lower("email"))
      .withColumn("country", F.upper("country"))
      .withColumn("signup_ts", timestamp("raw_signup_at")))

o = raw_copy(order_raw, order_cols)
for field in ["order_id", "customer_id", "status", "currency"]:
    o = o.withColumn(field, clean_text("raw_" + field))
o = o.withColumn("status", F.upper("status")).withColumn("currency", F.upper("currency"))
for src, target in [("ordered_at", "ordered_ts"), ("shipped_at", "shipped_ts"), ("ingested_at", "ingested_ts")]:
    o = o.withColumn(target, timestamp("raw_" + src))
o = o.withColumn("quantity", F.expr("try_cast(raw_quantity as int)"))
for field in ["unit_price", "discount", "total"]:
    o = o.withColumn(field, F.expr(f"try_cast(raw_{field} as decimal(12,2))"))

o.select("source_row_id", "raw_quantity", "quantity", "raw_unit_price", "unit_price").show(24)


## 7. Test one expectation interactively

The quantity range expectation detects the zero quantity. A null parsed quantity needs its own completeness rule: range checks alone are not sufficient.


In [ ]:
batch = order_batch.get_batch(batch_parameters={"dataframe": o})
single = batch.validate(gx.expectations.ExpectColumnValuesToBeBetween(
    column="quantity", min_value=1, max_value=100), result_format="BASIC")
print("Quantity range passed:", single.success)
print("Unexpected count:", single.result.get("unexpected_count"))
assert not single.success


## 8. Customer quality: missing values, formats and duplicate groups

**Problem:** missing and whitespace-only IDs, invalid email syntax, unsupported countries, impossible/future signup dates, repeated IDs, and repeated emails after normalization.

**Solution:** express each business rule once as a Spark Boolean column. GX checks that every value is `True`; the same flags produce quarantine reason codes. This keeps validation and row routing aligned. `coalesce(..., False)` treats an unknown result as failure, avoiding SQL three-valued-logic gaps.

All rows belonging to a duplicate group are rejected. Choosing the first row arbitrarily could associate orders with the wrong customer. The email regex checks shape, not mailbox existence; email uniqueness is a business assumption for this example.


In [ ]:
AS_OF_TS = F.to_timestamp(F.lit(AS_OF))
EMAIL_PATTERN = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
MONEY_PATTERN = r"^-?[0-9]+(\.[0-9]{1,2})?$"

def add_flags(df, rules):
    for name, condition in rules.items():
        df = df.withColumn("dq_" + name, F.coalesce(condition, F.lit(False)))
    return df

c = (c.withColumn("id_count", F.count(F.lit(1)).over(Window.partitionBy("customer_id")))
      .withColumn("email_count", F.count(F.lit(1)).over(Window.partitionBy("email"))))
customer_rules = {
    "customer_id_present": F.col("customer_id").isNotNull(),
    "customer_id_format": F.col("customer_id").rlike(r"^C[0-9]{3}$"),
    "customer_id_unique": F.col("id_count") == 1,
    "email_present": F.col("email").isNotNull(),
    "email_format": F.col("email").rlike(EMAIL_PATTERN),
    "email_unique": F.col("email_count") == 1,
    "country_allowed": F.col("country").isin("IN", "US", "GB"),
    "signup_parseable": F.col("signup_ts").isNotNull(),
    "signup_not_future": F.col("signup_ts") <= AS_OF_TS,
}
c_checked = add_flags(c, customer_rules).cache()

def with_reasons(df, rules):
    reasons = F.array(*[F.when(~F.col("dq_" + name), F.lit(name)) for name in rules])
    return (df.withColumn("dq_reasons", F.filter(reasons, lambda x: x.isNotNull()))
              .withColumn("dq_run_id", F.lit(RUN_ID)))

c_routed = with_reasons(c_checked, customer_rules)
c_good = c_routed.filter(F.size("dq_reasons") == 0).cache()
c_bad = c_routed.filter(F.size("dq_reasons") > 0).cache()
c_bad.select("source_row_id", "customer_id", "dq_reasons").show(truncate=False)
assert c_good.count() == 3
assert c_bad.count() == 12


## 9. Orders: relationships, conditional rules, arithmetic and freshness

**Problem:** an order may refer to an absent customer or to a customer rejected by our earlier checks. Other failures involve status/currency, malformed numbers, negative values, excessive discounts, incorrect totals, invalid dates, missing shipment dates, impossible event order, or stale ingestion.

**Solution:** join against distinct staging customer IDs and accepted customer IDs separately. The first check measures source referential integrity; the second prevents rejected parents from reaching trusted outputs. Join to accepted customers for signup and country rules. Distinct parent keys prevent joins from multiplying order rows.

The order contract requires quantity 1–100, unit price 0.01–10,000, nonnegative discount no greater than gross value, and exact decimal arithmetic. A `SHIPPED` order needs a shipment timestamp. Any supplied shipment timestamp must parse and lie between order time and the batch clock. Orders cannot precede signup. Ingestion must occur after the order and within the preceding 24 hours. Currency must match the customer's country under this store's single-currency-per-country policy.


In [ ]:
staged_ids = c.select("customer_id").filter(F.col("customer_id").isNotNull()).distinct().withColumn("parent_staged", F.lit(True))
accepted_parents = c_good.select("customer_id", "signup_ts", "country").withColumn("parent_accepted", F.lit(True))
o = (o.withColumn("id_count", F.count(F.lit(1)).over(Window.partitionBy("order_id")))
      .join(staged_ids, "customer_id", "left")
      .join(accepted_parents, "customer_id", "left"))
expected_currency = (F.when(F.col("country") == "IN", "INR")
                     .when(F.col("country") == "US", "USD")
                     .when(F.col("country") == "GB", "GBP"))
gross = F.col("quantity") * F.col("unit_price")
order_rules = {
    "order_id_present": F.col("order_id").isNotNull(),
    "order_id_format": F.col("order_id").rlike(r"^O[0-9]{3}$"),
    "order_id_unique": F.col("id_count") == 1,
    "customer_id_present": F.col("customer_id").isNotNull(),
    "customer_exists_in_staging": F.col("parent_staged") == True,
    "customer_accepted": F.col("parent_accepted") == True,
    "quantity_integer": F.trim(F.col("raw_quantity")).rlike(r"^[0-9]+$") & F.col("quantity").isNotNull(),
    "quantity_range": F.col("quantity").between(1, 100),
    "unit_price_range": F.col("unit_price").between(0.01, 10000),
    "discount_range": (F.col("discount") >= 0) & (F.col("discount") <= gross),
    "total_nonnegative": F.col("total") >= 0,
    "total_reconciles": F.col("total") == gross - F.col("discount"),
    "status_allowed": F.col("status").isin("PAID", "PENDING", "SHIPPED", "CANCELLED"),
    "currency_allowed": F.col("currency").isin("INR", "USD", "GBP"),
    "currency_matches_customer": F.col("currency") == expected_currency,
    "order_parseable": F.col("ordered_ts").isNotNull(),
    "order_not_future": F.col("ordered_ts") <= AS_OF_TS,
    "order_after_signup": F.col("ordered_ts") >= F.col("signup_ts"),
    "shipment_parseable_if_supplied": clean_text("raw_shipped_at").isNull() | F.col("shipped_ts").isNotNull(),
    "shipped_requires_timestamp": (F.col("status") != "SHIPPED") | F.col("shipped_ts").isNotNull(),
    "shipment_chronology": F.col("shipped_ts").isNull() | ((F.col("shipped_ts") >= F.col("ordered_ts")) & (F.col("shipped_ts") <= AS_OF_TS)),
    "ingestion_parseable": F.col("ingested_ts").isNotNull(),
    "ingestion_after_order": F.col("ingested_ts") >= F.col("ordered_ts"),
    "ingestion_fresh": F.col("ingested_ts").between(AS_OF_TS - F.expr("INTERVAL 24 HOURS"), AS_OF_TS),
}
for field in ["unit_price", "discount", "total"]:
    order_rules[field + "_money_parseable"] = (F.trim(F.col("raw_" + field)).rlike(MONEY_PATTERN)
                                               & F.col(field).isNotNull())
o_checked = add_flags(o, order_rules).cache()
o_routed = with_reasons(o_checked, order_rules)
o_good = o_routed.filter(F.size("dq_reasons") == 0).cache()
o_bad = o_routed.filter(F.size("dq_reasons") > 0).cache()
o_bad.select("source_row_id", "order_id", "dq_reasons").show(30, truncate=False)
assert o_checked.count() == SOURCE_MANIFEST["orders"], "A join unexpectedly changed row count."
assert o_good.count() == 3
assert o_bad.count() == 21


## 10. Build reusable GX suites and inspect failed rules

The Boolean checks cover complex business logic. We also use native GX expectations for nonempty batches, source record uniqueness, required IDs, unique business keys, and numeric ranges. Native expectations are convenient for standard column checks; Boolean flags make conditional and joined checks easy to inspect without custom GX plugins.

We validate the faulty batch first, then the accepted subset. An empty accepted batch must not pass just because it contains no bad values: every suite has a minimum row count of one. `BASIC` results may include example values; we persist a sanitized report later, rather than raw result samples that could expose customer data.


In [ ]:
def make_validation(name, batch_definition, rules, key, extras=()):
    suite = context.suites.add(gx.ExpectationSuite(name=name))
    expectations = [
        gx.expectations.ExpectTableRowCountToBeBetween(min_value=1, max_value=100000),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="source_row_id"),
        gx.expectations.ExpectColumnValuesToBeUnique(column="source_row_id"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column=key),
        gx.expectations.ExpectColumnValuesToBeUnique(column=key),
    ]
    expectations += [gx.expectations.ExpectColumnValuesToBeInSet(
        column="dq_" + rule, value_set=[True]) for rule in rules]
    for expectation in [*expectations, *extras]:
        suite.add_expectation(expectation)
    definition = context.validation_definitions.add(gx.ValidationDefinition(
        name="validate_" + name, data=batch_definition, suite=suite))
    return suite, definition

customer_suite, customer_validation = make_validation("customer_contract", customer_batch, customer_rules, "customer_id")
order_suite, order_validation = make_validation("order_contract", order_batch, order_rules, "order_id", [
    gx.expectations.ExpectColumnValuesToNotBeNull(column="quantity"),
    gx.expectations.ExpectColumnValuesToBeBetween(column="quantity", min_value=1, max_value=100),
])
raw_customer_result = validate("staged_customers", customer_validation, c_checked)
raw_order_result = validate("staged_orders", order_validation, o_checked)
assert not raw_customer_result.success and not raw_order_result.success
good_customer_result = validate("accepted_customers", customer_validation, c_good)
good_order_result = validate("accepted_orders", order_validation, o_good)
assert good_customer_result.success and good_order_result.success


## 11. Warning thresholds are different from rejection rules

**Problem:** optional phone coverage is low, but missing phones do not invalidate orders. Treating every issue as fatal stops useful work; making every rule tolerant hides serious defects.

**Solution:** run phone coverage as a separate warning with `mostly=0.80`. Only one of three accepted customers has a phone, so the warning fails, while customers remain accepted. Critical rules such as IDs and parent validity retain 100% requirements. A mostly threshold evaluates a proportion, not a fixed number of tolerated rows.


In [ ]:
warning_batch = customer_batch.get_batch(batch_parameters={"dataframe": c_good})
phone_warning = warning_batch.validate(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="phone", mostly=0.80),
    result_format="BASIC")
print("Phone coverage meets 80%:", phone_warning.success)
assert not phone_warning.success
print("Action: warn the source owner; do not reject otherwise valid customers.")


## 12. Inspect quarantine and prove that no source rows disappeared

**Problem:** counting rule failures can overstate rejected rows because one row may fail several rules. Joins and filters can also accidentally lose or duplicate data.

**Solution:** count rejected records separately from exploded reason counts. Prove raw = accepted + rejected, source record IDs remain unique, and no accepted order points to a rejected customer. Keep all reasons on each bad row. Raw evidence belongs in restricted quarantine storage in a real deployment.


In [ ]:
counts = {}
for name, raw, good, bad in [("customers", customer_raw, c_good, c_bad), ("orders", order_raw, o_good, o_bad)]:
    incoming, accepted, rejected = raw.count(), good.count(), bad.count()
    counts[name] = {"incoming": incoming, "good": accepted, "bad": rejected,
                    "reject_rate": rejected / incoming if incoming else 1.0}
    combined = good.select("source_row_id").unionByName(bad.select("source_row_id"))
    assert incoming == accepted + rejected
    assert combined.distinct().count() == incoming
    assert raw.select("source_row_id").exceptAll(combined).count() == 0
    assert combined.exceptAll(raw.select("source_row_id")).count() == 0
    print(name, counts[name])
    bad.select(F.explode("dq_reasons").alias("reason")).groupBy("reason").count().orderBy(F.desc("count")).show(40, truncate=False)

assert o_good.join(c_good.select("customer_id"), "customer_id", "left_anti").count() == 0


## 13. Batch health: reconcile totals and reject-rate budgets

**Problem:** three clean orders can pass row checks even when most of the delivery is broken. Aggregate checks describe whether the batch is useful enough to publish.

**Solution:** build a one-row metrics DataFrame and validate it with GX. Reconcile totals **by currency**, since adding INR and USD produces a meaningless number. The accepted orders should total INR 380.00 and USD 190.00, with zero arithmetic residual for each currency.

The operational example permits at most **10% rejection** for each dataset. Our deliberately faulty batch exceeds this, so the gate blocks normal publication. A separately named **demonstration policy** permits up to 90% rejection so we can finish the good/bad HDFS walkthrough without pretending the batch is operationally healthy. Both policies still require nonempty accepted data and passing row contracts. In an actual job, select an approved policy once; do not relax a threshold automatically after a failure.


In [ ]:
reconciliation = o_good.groupBy("currency").agg(
    F.sum("total").alias("total"),
    F.sum(F.col("quantity") * F.col("unit_price") - F.col("discount")).alias("expected_total")
).withColumn("residual", F.col("total") - F.col("expected_total"))
reconciliation.show()
reconciliation_ok = reconciliation.filter(F.col("residual") != 0).count() == 0
assert reconciliation_ok
assert {r.currency: str(r.total) for r in reconciliation.collect()} == {"INR": "380.00", "USD": "190.00"}

metrics = spark.createDataFrame([(
    counts["customers"]["reject_rate"], counts["orders"]["reject_rate"],
    counts["customers"]["good"], counts["orders"]["good"], reconciliation_ok,
)], "customer_reject_rate double, order_reject_rate double, good_customers long, good_orders long, reconciles boolean")
metrics_batch = source.add_dataframe_asset(name="batch_metrics").add_batch_definition_whole_dataframe("whole")
def metric_contract(name, max_reject_rate):
    suite = context.suites.add(gx.ExpectationSuite(name=name))
    for field in ["customer_reject_rate", "order_reject_rate"]:
        suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column=field, min_value=0, max_value=max_reject_rate))
    for field in ["good_customers", "good_orders"]:
        suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column=field, min_value=1))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(column="reconciles", value_set=[True]))
    return context.validation_definitions.add(gx.ValidationDefinition(name="validate_" + name, data=metrics_batch, suite=suite))

operational_definition = metric_contract("operational_batch_policy", 0.10)
demo_definition = metric_contract("demonstration_batch_policy", 0.90)
operational_result = validate("operational_batch", operational_definition, metrics)
assert not operational_result.success
print("Operational publication BLOCKED: excessive rejection.")
demo_result = validate("demonstration_batch", demo_definition, metrics)
assert demo_result.success


## 14. Run a Checkpoint and enforce an explicit publication gate

A Checkpoint provides a reusable runner around a Validation Definition. This checkpoint validates the accepted orders. The gate also requires accepted customers, both delivery contracts, reconciliation, and the explicitly selected demonstration policy.

The two datasets use different DataFrames, so their Validation Definitions are run separately above. Passing a single DataFrame parameter to a multi-definition checkpoint would not supply a different DataFrame to each table.


In [ ]:
checkpoint = context.checkpoints.add(gx.Checkpoint(
    name="accepted_orders_checkpoint", validation_definitions=[order_validation], result_format="BASIC"))
checkpoint_result = checkpoint.run(batch_parameters={"dataframe": o_good})
assert checkpoint_result.success

def enforce_gate(checks):
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise RuntimeError("Publication blocked: " + ", ".join(failed))

try:
    enforce_gate({"operational_reject_budget": operational_result.success})
except RuntimeError as error:
    print("Expected gate failure:", error)

PUBLICATION_POLICY = "demonstration_batch_policy"
enforce_gate({
    "customer_delivery": customer_delivery_result.success,
    "order_delivery": order_delivery_result.success,
    "customer_types": types_match(customer_raw, customer_cols),
    "order_types": types_match(order_raw, order_cols),
    "accepted_customers": good_customer_result.success,
    "accepted_orders_checkpoint": checkpoint_result.success,
    "demonstration_budget": demo_result.success,
    "currency_reconciliation": reconciliation_ok,
})
print("Demonstration publication gate passed.")


## 15. Persist accepted rows, quarantine and audit evidence in HDFS

**Problem:** validation printed in a notebook is not durable evidence. Multiple output writes can fail partway through a run.

**Solution:** write into this run's unique directory and create a publication marker only after all datasets and audits have been written and read back successfully. Consumers should discover runs through `published`, never by listing `good` directories alone. This is a small file-based publication protocol, not a transactional table format or a concurrency-safe production orchestrator.

Good data contains typed business columns and source lineage. Bad data also contains raw values, flags, and all reason codes. Audit reports omit row samples and preserve aggregate validation outcomes. The saved suites document the rules used in this run; the ephemeral GX context itself is not restored automatically from these files.


In [ ]:
customer_output = c_good.select("source_row_id", "customer_id", "email", "country", "signup_ts", "phone", "dq_run_id")
order_output = o_good.select("source_row_id", "order_id", "customer_id", "ordered_ts", "shipped_ts",
                             "quantity", "unit_price", "discount", "total", "status", "currency", "ingested_ts", "dq_run_id")
outputs = {"good/customers": customer_output, "bad/customers": c_bad,
           "good/orders": order_output, "bad/orders": o_bad}
for relative_path, df in outputs.items():
    df.write.mode("errorifexists").parquet(f"{RUN_ROOT}/{relative_path}")

def write_json_document(relative_path, document):
    # A Spark text output directory contains one JSON document in its part file.
    payload = json.dumps(document, default=str, sort_keys=True)
    spark.createDataFrame([(payload,)], "value string").coalesce(1).write.mode("errorifexists").text(f"{RUN_ROOT}/{relative_path}")

def sanitized_result(result):
    return {"success": bool(result.success), "statistics": result.statistics,
            "expectations": [{"type": item.expectation_config.type,
                              "column": item.expectation_config.kwargs.get("column"),
                              "success": bool(item.success),
                              "unexpected_count": item.result.get("unexpected_count")}
                             for item in result.results]}

audit = {"run_id": RUN_ID, "business_as_of_utc": AS_OF, "publication_policy": PUBLICATION_POLICY,
         "operational_policy_passed": bool(operational_result.success), "counts": counts,
         "phone_warning_passed": bool(phone_warning.success),
         "validations": {label: sanitized_result(result) for label, result in validation_results.items()}}
write_json_document("audit/results", audit)
write_json_document("audit/suites", [suite.to_json_dict() for suite in context.suites.all()])

# Verify full row contents (including duplicate multiplicity), not just write completion.
for relative_path, original in outputs.items():
    restored = spark.read.parquet(f"{RUN_ROOT}/{relative_path}").select(*original.columns)
    assert original.exceptAll(restored).count() == 0
    assert restored.exceptAll(original).count() == 0
stored_audit = json.loads(spark.read.text(f"{RUN_ROOT}/audit/results").first().value)
assert stored_audit["run_id"] == RUN_ID
assert len(json.loads(spark.read.text(f"{RUN_ROOT}/audit/suites").first().value)) >= 6

write_json_document("published", {"run_id": RUN_ID, "policy": PUBLICATION_POLICY,
                                 "good_customers": f"{RUN_ROOT}/good/customers",
                                 "good_orders": f"{RUN_ROOT}/good/orders", "counts": counts})
print("Published demonstration run:", RUN_ROOT)


## 16. Read the published run as a consumer

Only a run with a completed publication marker is eligible. SQL temporary views let us query accepted data without introducing more staging tables or depending on Hive. The `gx` database still contains just the two staging tables.

Expected outcome: **3 accepted / 12 rejected customers** and **3 accepted / 21 rejected orders**. Operational quality remains failed; the marker explicitly records the demonstration policy.


In [ ]:
published = json.loads(spark.read.text(f"{RUN_ROOT}/published").first().value)
assert published["policy"] == "demonstration_batch_policy"
spark.read.parquet(published["good_customers"]).createOrReplaceTempView("accepted_customers")
spark.read.parquet(published["good_orders"]).createOrReplaceTempView("accepted_orders")
spark.sql("""
    SELECT o.currency, COUNT(*) AS orders, SUM(o.total) AS order_value
    FROM accepted_orders o JOIN accepted_customers c ON o.customer_id = c.customer_id
    GROUP BY o.currency ORDER BY o.currency
""").show()
spark.read.parquet(f"{RUN_ROOT}/bad/orders").select("source_row_id", "order_id", "dq_reasons").show(30, truncate=False)
print("Inspect from a WSL terminal:")
print(f"hdfs dfs -ls -R {RUN_ROOT}")


## 17. Correct a rejected record without losing the evidence

**Problem:** fixing `quantity='two'` should not silently modify the original quarantine or publish an unvalidated replacement.

**Solution:** demonstrate the corrected value in an isolated probe, preserving its source record ID. Both parseability and range must pass. A real replay must then rerun **all** order checks, including customer eligibility, totals, uniqueness across the target, and freshness under the appropriate batch clock. This probe is not appended to good data.

The original run directories are immutable in this notebook. Replay uses a new run ID; repeated business IDs need an explicit upsert or deduplication policy before production ingestion.


In [ ]:
repair = (o_bad.filter(F.col("source_row_id") == "o07")
          .select("source_row_id", "raw_quantity")
          .withColumn("corrected_raw_quantity", F.lit("2"))
          .withColumn("quantity", F.expr("try_cast(corrected_raw_quantity as int)")))
repair_batch = order_batch.get_batch(batch_parameters={"dataframe": repair})
for expectation in [
    gx.expectations.ExpectColumnValuesToNotBeNull(column="quantity"),
    gx.expectations.ExpectColumnValuesToBeBetween(column="quantity", min_value=1, max_value=100),
]:
    assert repair_batch.validate(expectation, result_format="BASIC").success
repair.show()
assert o_bad.filter(F.col("source_row_id") == "o07").select("raw_quantity").first()[0] == "two"


## 18. Practice and operating decisions

1. Change an order to `currency='USD'` for an Indian customer. Both currencies are allowed, but which relationship rule fails?
2. Add a customer with a valid-looking email and a duplicate normalized email. Why must both records be investigated?
3. Put `Infinity`, a huge overflow value, or `100.001` in a money field. Inspect raw text, parsed value, and reasons.
4. Send only orders with rejected customer IDs. Confirm the accepted-row minimum blocks an empty publication.
5. Add an unexpected source column. Decide whether this is a breaking change or an approved contract revision.
6. Improve phone coverage to two out of three customers. Explain why an 80% requirement still fails.

After modifying source fixtures, restart and run all cells with a new run ID. Update the independent manifest and expected assertions only when the changed delivery is intentional. Do not change thresholds solely to make a failed batch pass.

For larger datasets, avoid collecting bad rows to the driver; write distributed quarantine as shown. Cache only reused data and unpersist it afterward. Production deployments need versioned contracts, persistent validation configuration, approved freshness windows, restricted customer-data access, and a retention policy. Source manifests should include business control totals as well as row counts; a row count alone cannot detect every missing/replaced record. Monitoring distribution changes against historical baselines is an additional batch-level pattern, separate from the deterministic row contracts demonstrated here.

Official API references: [GX Spark DataFrames](https://docs.greatexpectations.io/docs/core/connect_to_data/dataframes/), [GX Validation Definitions](https://docs.greatexpectations.io/docs/core/run_validations/run_a_validation_definition/), and [Spark safe timestamp parsing](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.functions.try_to_timestamp.html).


In [ ]:
for df in [c_checked, c_good, c_bad, o_checked, o_good, o_bad]:
    df.unpersist()
print("Cached teaching datasets released. HDFS run files remain available.")
# Optional when finished: spark.stop()
